# Build CENTRAL Dataset

Notebook นี้เป็นตัวสร้างข้อมูลกลางที่ runner หลักเรียกใช้ รับผล harmonization จาก dataset ทั้งสี่ รวม labels และ metadata แยกจาก features แล้วสร้างชุด train, validation และ test

Input: data-pipeline/sub-dataset/<dataset>/output/harmonized/
Output: data-pipeline/CENTRAL/output/harmonized/ และ data-pipeline/CENTRAL/output/splits/

Notebook นี้เตรียมและแบ่งข้อมูล ไม่ได้ฝึกโมเดลหรือทำ SMOTE; Borderline-SMOTE อยู่ใน borderline_smote.ipynb


## 1. ตั้งค่าโฟลเดอร์และชุดข้อมูล

หา project root จากตำแหน่งที่เปิด notebook ได้ และกำหนดชุดข้อมูลต้นทางที่ต้องมีครบ


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_project_root(start=None):
    origin = Path(start or Path.cwd()).resolve()
    for candidate in (origin, *origin.parents):
        central_notebook = candidate / "data-pipeline" / "CENTRAL" / "build_dataset.ipynb"
        if central_notebook.is_file():
            return candidate
        if candidate.name == "CENTRAL" and (candidate / "build_dataset.ipynb").is_file():
            if candidate.parent.name == "data-pipeline":
                return candidate.parent.parent
    raise FileNotFoundError(f"หา project root จากตำแหน่งนี้ไม่พบ: {origin}")


PROJECT_ROOT = find_project_root()
PIPELINE_DIR = PROJECT_ROOT / "data-pipeline"
CENTRAL_DIR = PIPELINE_DIR / "CENTRAL"
DATASETS_DIR = PIPELINE_DIR / "sub-dataset"
DATASETS = ("NSL-KDD", "UNSW-NB15", "CIC-IDS2017", "BoT-IoT")
SEED = 42
VALIDATION_FRACTION = 0.20

print("Project:", PROJECT_ROOT)
print("Central output:", CENTRAL_DIR / "output")
print("Datasets:", ", ".join(DATASETS))

## 2. อ่านและตรวจตาราง harmonized

แต่ละชุดต้องมี features, labels และ metadata จำนวนแถวเท่ากัน และต้องเรียง record_id ตรงกัน


In [ ]:
feature_parts, label_parts, metadata_parts = [], [], []

for name in DATASETS:
    dataset_dir = DATASETS_DIR / name / "output" / "harmonized"
    paths = {table: dataset_dir / f"{table}.csv" for table in ("features", "labels", "metadata")}
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"{name} ยังไม่มีผล harmonization ครบ: {missing}")

    features = pd.read_csv(paths["features"], low_memory=False)
    labels = pd.read_csv(paths["labels"], low_memory=False)
    metadata = pd.read_csv(paths["metadata"], low_memory=False)
    if not len(features) == len(labels) == len(metadata):
        raise ValueError(f"จำนวนแถวไม่ตรงกันใน {name}: features={len(features)}, labels={len(labels)}, metadata={len(metadata)}")
    if not features["record_id"].astype(str).reset_index(drop=True).equals(labels["record_id"].astype(str).reset_index(drop=True)):
        raise ValueError(f"record_id ของ features และ labels ไม่ตรงกันใน {name}")
    if not features["record_id"].astype(str).reset_index(drop=True).equals(metadata["record_id"].astype(str).reset_index(drop=True)):
        raise ValueError(f"record_id ของ features และ metadata ไม่ตรงกันใน {name}")

    print(f"{name}: {len(features):,} แถว")
    feature_parts.append(features)
    label_parts.append(labels)
    metadata_parts.append(metadata)

features = pd.concat(feature_parts, ignore_index=True)
labels = pd.concat(label_parts, ignore_index=True)
metadata = pd.concat(metadata_parts, ignore_index=True)

if not features["record_id"].is_unique:
    raise ValueError("พบ record_id ซ้ำข้าม dataset")
print("รวมก่อนแบ่งชุด:", f"{len(features):,}", "แถว")

## 3. รวมป้ายกำกับเป็น 6 กลุ่ม

เก็บคำตอบดั้งเดิมและชื่อหมวดเดิมไว้ แล้วเพิ่มรหัสกลุ่มกลางใน labels.csv; labels จะไม่ถูกรวมเข้า features


In [ ]:
mitre_map = {
    "normal": (0, "Benign / Normal", "Baseline Operations"),
    "benign": (0, "Benign / Normal", "Baseline Operations"),
    "dos": (1, "Denial of Service (DoS)", "TA0040: Impact"),
    "dos/ddos": (1, "Denial of Service (DoS)", "TA0040: Impact"),
    "ddos": (1, "Denial of Service (DoS)", "TA0040: Impact"),
    "probe": (2, "Reconnaissance & Probing", "TA0043/TA0007: Recon & Discovery"),
    "probe/recon": (2, "Reconnaissance & Probing", "TA0043/TA0007: Recon & Discovery"),
    "reconnaissance": (2, "Reconnaissance & Probing", "TA0043/TA0007: Recon & Discovery"),
    "analysis": (2, "Reconnaissance & Probing", "TA0043/TA0007: Recon & Discovery"),
    "exploits": (3, "Exploits & PrivEsc", "TA0001/TA0004: Initial Access & PrivEsc"),
    "exploit/u2r": (3, "Exploits & PrivEsc", "TA0001/TA0004: Initial Access & PrivEsc"),
    "shellcode": (3, "Exploits & PrivEsc", "TA0001/TA0004: Initial Access & PrivEsc"),
    "u2r": (3, "Exploits & PrivEsc", "TA0001/TA0004: Initial Access & PrivEsc"),
    "r2l": (4, "Credential Abuse (R2L)", "TA0006: Credential Access"),
    "brute-force/r2l": (4, "Credential Abuse (R2L)", "TA0006: Credential Access"),
    "generic": (5, "Malware, C2 & Botnet", "TA0011/TA0008: C2 & Lateral Movement"),
    "generic/fuzzer": (5, "Malware, C2 & Botnet", "TA0011/TA0008: C2 & Lateral Movement"),
    "fuzzers": (5, "Malware, C2 & Botnet", "TA0011/TA0008: C2 & Lateral Movement"),
    "backdoor": (5, "Malware, C2 & Botnet", "TA0011/TA0008: C2 & Lateral Movement"),
    "botnet/backdoor": (5, "Malware, C2 & Botnet", "TA0011/TA0008: C2 & Lateral Movement"),
    "worms": (5, "Malware, C2 & Botnet", "TA0011/TA0008: C2 & Lateral Movement"),
}

normalized_source = labels["label_source_category"].astype(str).str.strip().str.lower()
mitre_info = [
    mitre_map.get(value, (5, "Malware, C2 & Botnet", "TA0011/TA0008: C2 & Lateral Movement"))
    for value in normalized_source
]
labels["attack_category_id"] = [item[0] for item in mitre_info]
labels["attack_category"] = [item[1] for item in mitre_info]
labels["mitre_tactic"] = [item[2] for item in mitre_info]
labels["label_binary"] = (labels["attack_category_id"] > 0).astype(int)
labels = labels[[
    "record_id", "label_raw", "label_source_category", "attack_category",
    "attack_category_id", "label_binary", "mitre_tactic",
]]

if not features["record_id"].astype(str).reset_index(drop=True).equals(labels["record_id"].astype(str).reset_index(drop=True)):
    raise ValueError("record_id ของ features และ labels ไม่ตรงกันหลังรวม")

## 4. แบ่ง Train, Validation และ Test

คง train/test ที่มาจากต้นฉบับไว้ แล้วกัน 20% ของแต่ละ source train เป็น validation ด้วย seed คงที่


In [ ]:
def assign_splits(metadata: pd.DataFrame, seed: int = 42, validation_fraction: float = 0.20) -> pd.Series:
    result = pd.Series(pd.NA, index=metadata.index, dtype="string")
    for _, indexes in metadata.groupby("source_dataset", sort=True).groups.items():
        block = metadata.loc[indexes]
        official = block["source_split"].astype("string").str.lower()
        train_indices = block.index[official.isin(["train", "kddtrain"])].to_numpy()
        test_indices = block.index[official.isin(["test", "kddtest"])].to_numpy()

        rng = np.random.default_rng(seed)
        shuffled = rng.permutation(train_indices)
        validation_size = int(np.floor(len(shuffled) * validation_fraction))
        validation_indices = shuffled[:validation_size]
        train_indices = shuffled[validation_size:]

        result.loc[train_indices] = "train"
        result.loc[validation_indices] = "validation"
        result.loc[test_indices] = "test"
    return result


metadata["data_split"] = assign_splits(
    metadata, seed=SEED, validation_fraction=VALIDATION_FRACTION
)
if metadata["data_split"].isna().any():
    unassigned = int(metadata["data_split"].isna().sum())
    raise ValueError(f"มี {unassigned:,} แถวที่กำหนด split ไม่ได้")

rows_per_source_split = metadata.groupby(["source_dataset", "data_split"])["record_id"].transform("size")
metadata["sample_weight"] = 1.0 / rows_per_source_split.astype(float)
print(pd.crosstab(metadata["source_dataset"], metadata["data_split"], margins=True))

## 5. บันทึกตารางและรายงาน

ทุก split มี features.csv, labels.csv และ metadata.csv เรียงตาม record_id เดียวกัน; test ไม่ผ่านขั้น SMOTE


In [ ]:
output_dir = CENTRAL_DIR / "output"
harmonized_dir = output_dir / "harmonized"
harmonized_dir.mkdir(parents=True, exist_ok=True)
features.to_csv(harmonized_dir / "features.csv", index=False)
labels.to_csv(harmonized_dir / "labels.csv", index=False)
metadata.to_csv(harmonized_dir / "metadata.csv", index=False)

report = pd.DataFrame([
    {"table": "features", "rows": len(features), "columns": len(features.columns)},
    {"table": "labels", "rows": len(labels), "columns": len(labels.columns)},
    {"table": "metadata", "rows": len(metadata), "columns": len(metadata.columns)},
])
report.to_csv(harmonized_dir / "harmonization_report.csv", index=False)

split_masks = {
    "train": metadata["data_split"].eq("train"),
    "validation": metadata["data_split"].eq("validation"),
    "train_full": metadata["data_split"].isin(["train", "validation"]),
    "test": metadata["data_split"].eq("test"),
}
splits_dir = output_dir / "splits"
for split_name, mask in split_masks.items():
    split_dir = splits_dir / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    features.loc[mask].to_csv(split_dir / "features.csv", index=False)
    labels.loc[mask].to_csv(split_dir / "labels.csv", index=False)
    metadata.loc[mask].to_csv(split_dir / "metadata.csv", index=False)
    print(f"{split_name}: {int(mask.sum()):,} แถว -> {split_dir}")

print()
print("CENTRAL dataset build complete")
print("แถวทั้งหมด:", f"{len(features):,}")
print("แถวต่อ dataset:")
print(metadata["source_dataset"].value_counts().sort_index())
print("ผลลัพธ์:", output_dir)